In [ ]:
!pip install -q transformers datasets evaluate pycocotools accelerate timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [ ]:
# libraries
import torch
import os
import zipfile

import glob
import json
import os
from PIL import Image
import numpy as np

In [ ]:
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.11.0+cu130
GPU: Tesla T4


In [ ]:
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!pip install -q -U kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.0 MB/s eta 0:00:00


In [ ]:
!kaggle datasets download -d minhducnguyen9705/foggy-cityscapes

Dataset URL: https://www.kaggle.com/datasets/minhducnguyen9705/foggy-cityscapes
License(s): apache-2.0
100% 29.1G/29.1G [06:48<00:00, 76.5MB/s]



In [ ]:
!pip install appdirs tqdm requests

In [ ]:
!git clone https://github.com/mcordts/cityscapesScripts.git

Cloning into 'cityscapesScripts'...
remote: Enumerating objects: 688, done.
remote: Counting objects: 100% (260/260), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 688 (delta 204), reused 191 (delta 167), pack-reused 428 (from 1)
Receiving objects: 100% (688/688), 840.59 KiB | 8.76 MiB/s, done.
Resolving deltas: 100% (390/390), done.


In [ ]:
!python cityscapesScripts/cityscapesscripts/download/downloader.py \
    -d /content \
    gtBboxValidated_trainval.zip

Cityscapes username or email address: sheetals
Cityscapes password: 
Store credentials unencrypted in '/root/.local/share/__main__/credentials.json' [y/N]: N
Download progress:  99% 3.15M/3.20M [00:01<00:00, 2.62MB/s]


In [ ]:
!unzip -q /content/gtBboxValidated_trainval.zip -d /content/cityscapes

In [ ]:
zip_path = "/content/foggy-cityscapes.zip"
out_dir = "/content/foggy_cityscapes"

with zipfile.ZipFile(zip_path) as z:
    refined = z.read("foggy_trainval_refined_filenames.txt").decode().splitlines()

    selected = []
    for base in refined:
        for beta in ["0.005", "0.01", "0.02"]:
            path = f"leftImg8bit_foggy/{base}_leftImg8bit_foggy_beta_{beta}.png"
            if path in z.namelist():
                selected.append(path)

    print("Images to extract:", len(selected))

    for path in selected:
        z.extract(path, out_dir)

print("Done!")

Images to extract: 1650
Done!


In [ ]:
foggy = glob.glob("/content/foggy_cityscapes/**/*.png", recursive=True)

img_path = foggy[0]
filename = os.path.basename(img_path)
base = filename.split("_leftImg8bit_foggy_")[0]
city = os.path.basename(os.path.dirname(img_path))
split = os.path.basename(os.path.dirname(os.path.dirname(img_path)))

ann_path = f"/content/cityscapes/gtBboxValidated/{split}/{city}/{base}_gtBboxValidated.json"

image = Image.open(img_path)

with open(ann_path) as f:
    ann = json.load(f)

print("Foggy images:", len(foggy))
print("Image size:", image.size)
print("Objects:", len(ann["objects"]))

Foggy images: 1650
Image size: (2048, 1024)
Objects: 7


In [ ]:
from transformers import DetrImageProcessor

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

print("DETR processor loaded!")

preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

DETR processor loaded!


In [ ]:
import re
import json
import os
import glob
import torch
from PIL import Image
from torch.utils.data import Dataset

class FoggyCityscapesDataset(Dataset):
    def __init__(self, image_paths, processor):
        self.image_paths = image_paths
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        # Load image
        image = Image.open(img_path).convert("RGB")

        # Get split, city, and base filename
        filename = os.path.basename(img_path)
        base = filename.split("_leftImg8bit_foggy_")[0]
        city = os.path.basename(os.path.dirname(img_path))
        split = os.path.basename(os.path.dirname(os.path.dirname(img_path)))

        # Find matching annotation
        ann_path = (
            f"/content/cityscapes/gtBboxValidated/"
            f"{split}/{city}/{base}_gtBboxValidated.json"
        )

        with open(ann_path) as f:
            ann = json.load(f)

        # Get fog density
        beta = float(re.search(r"beta_(0\.\d+)", filename).group(1))

        # Parse objects
        boxes = []
        class_labels = []

        for obj in ann["objects"]:
            class_id = int(torch.tensor(obj["soft_label"]).argmax())

            # Skip cantsolve
            if class_id == 8:
                continue

            x1, y1, x2, y2 = obj["bbox"]

            boxes.append([x1, y1, x2, y2])
            class_labels.append(class_id)

        # DETR processor expects COCO-style annotations
        annotations = {
            "image_id": idx,
            "annotations": [
                {
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "category_id": label,
                    "area": (x2 - x1) * (y2 - y1),
                    "iscrowd": 0,
                }
                for (x1, y1, x2, y2), label in zip(boxes, class_labels)
            ],
        }

        encoding = self.processor(
            images=image,
            annotations=annotations,
            return_tensors="pt",
        )

        # Remove batch dimension
        encoding = {k: v.squeeze(0) if hasattr(v, "shape") and v.shape[0] == 1 else v
                    for k, v in encoding.items()}

        encoding["beta"] = torch.tensor(beta, dtype=torch.float32)

        return encoding

In [ ]:
train_images = glob.glob(
    "/content/foggy_cityscapes/leftImg8bit_foggy/train/**/*.png",
    recursive=True
)

val_images = glob.glob(
    "/content/foggy_cityscapes/leftImg8bit_foggy/val/**/*.png",
    recursive=True
)

train_dataset = FoggyCityscapesDataset(train_images, processor)
val_dataset = FoggyCityscapesDataset(val_images, processor)

print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))

Train images: 1494
Val images: 156


In [ ]:
from transformers import DetrForObjectDetection

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=8,
    ignore_mismatched_sizes=True
)

print("DETR loaded!")

config.json:   0%|          | 0.00/4.59k [00:00<?, ?B/s]

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `91`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  167MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |             

DETR loaded!


In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    pixel_mask = torch.stack([item["pixel_mask"] for item in batch])
    labels = [item["labels"][0] for item in batch]
    betas = torch.stack([item["beta"] for item in batch])

    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "labels": labels,
        "beta": betas
    }

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

print("DataLoaders ready!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

DataLoaders ready!
Training batches: 374
Validation batches: 39


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

batch = next(iter(train_loader))

pixel_values = batch["pixel_values"].to(device)
pixel_mask = batch["pixel_mask"].to(device)

labels = [
    {k: v.to(device) for k, v in target.items()}
    for target in batch["labels"]
]

outputs = model(
    pixel_values=pixel_values,
    pixel_mask=pixel_mask,
    labels=labels
)

print("Logits:", outputs.logits.shape)
print("Predicted boxes:", outputs.pred_boxes.shape)
print("Loss:", outputs.loss.item())

Logits: torch.Size([4, 100, 9])
Predicted boxes: torch.Size([4, 100, 4])
Loss: 3.277589797973633


In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm

optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

num_epochs = 3
best_val_loss = float("inf")

for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    train_loss = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in progress:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)

        labels = [
            {k: v.to(device) for k, v in target.items()}
            for target in batch["labels"]
        ]

        optimizer.zero_grad()

        outputs = model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    # ---- Validation ----
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(device)
            pixel_mask = batch["pixel_mask"].to(device)

            labels = [
                {k: v.to(device) for k, v in target.items()}
                for target in batch["labels"]
            ]

            outputs = model(
                pixel_values=pixel_values,
                pixel_mask=pixel_mask,
                labels=labels
            )

            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch {epoch+1}: "
        f"Train Loss = {avg_train_loss:.4f}, "
        f"Val Loss = {avg_val_loss:.4f}"
    )

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained("/content/detr_baseline_best")
        processor.save_pretrained("/content/detr_baseline_best")
        print("Saved best model!")

Epoch 1/3:   0%|          | 0/374 [00:00<?, ?it/s]

Epoch 1: Train Loss = 2.7181, Val Loss = 2.8108


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model!


Epoch 2/3:   0%|          | 0/374 [00:00<?, ?it/s]

...

Saved best model!


Epoch 3/3:   0%|          | 0/374 [00:00<?, ?it/s]

Epoch 3: Train Loss = 2.0231, Val Loss = 2.4706


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model!


In [ ]:
from transformers import DetrForObjectDetection, DetrImageProcessor

baseline_processor = DetrImageProcessor.from_pretrained(
    "/content/detr_baseline_best"
)

baseline_model = DetrForObjectDetection.from_pretrained(
    "/content/detr_baseline_best"
).to(device)

baseline_model.eval()

print("Best baseline loaded!")

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

Best baseline loaded!


In [ ]:
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Running validation"):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)

        outputs = baseline_model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask
        )

        all_predictions.append({
            "logits": outputs.logits.cpu(),
            "boxes": outputs.pred_boxes.cpu()
        })

        all_targets.extend(batch["labels"])

print("Validation inference complete!")
print("Batches:", len(all_predictions))
print("Images:", len(all_targets))

Running validation:   0%|          | 0/39 [00:00<?, ?it/s]

Validation inference complete!
Batches: 39
Images: 156


In [ ]:
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 30.7 MB/s eta 0:00:00


In [ ]:
from torchmetrics.detection import MeanAveragePrecision

metric = MeanAveragePrecision(
    box_format="cxcywh",
    iou_type="bbox"
)

for batch_preds, target in zip(all_predictions, [
    all_targets[i:i+4] for i in range(0, len(all_targets), 4)
]):
    probs = batch_preds["logits"].softmax(-1)

    for i in range(len(target)):
        # Remove DETR's "no object" class
        scores, labels = probs[i, :, :-1].max(dim=-1)
        boxes = batch_preds["boxes"][i]

        # Keep reasonably confident detections
        keep = scores > 0.5

        prediction = {
            "boxes": boxes[keep],
            "scores": scores[keep],
            "labels": labels[keep]
        }

        ground_truth = {
            "boxes": target[i]["boxes"],
            "labels": target[i]["class_labels"]
        }

        metric.update([prediction], [ground_truth])

results = metric.compute()

print("mAP:", results["map"].item())
print("mAP@50:", results["map_50"].item())
print("mAP@75:", results["map_75"].item())

mAP: 0.030721120536327362
mAP@50: 0.058638088405132294
mAP@75: 0.029604995623230934


In [ ]:
from torchmetrics.detection import MeanAveragePrecision

metric = MeanAveragePrecision(
    box_format="xyxy",
    iou_type="bbox"
)

for batch_preds, target in zip(
    all_predictions,
    [all_targets[i:i+4] for i in range(0, len(all_targets), 4)]
):
    probs = batch_preds["logits"].softmax(-1)

    for i in range(len(target)):
        # DETR boxes: normalized cx, cy, width, height
        boxes = batch_preds["boxes"][i].clone()

        # Convert to absolute xyxy coordinates
        h, w = target[i]["orig_size"]

        cx, cy, bw, bh = boxes.unbind(-1)

        x1 = (cx - bw / 2) * w
        y1 = (cy - bh / 2) * h
        x2 = (cx + bw / 2) * w
        y2 = (cy + bh / 2) * h

        boxes_xyxy = torch.stack([x1, y1, x2, y2], dim=-1)

        # Remove "no object" class, but keep ALL predictions
        scores, labels = probs[i, :, :-1].max(dim=-1)

        prediction = {
            "boxes": boxes_xyxy,
            "scores": scores,
            "labels": labels
        }

        # Ground truth: normalized cxcywh -> absolute xyxy
        gt_boxes = target[i]["boxes"].clone()

        gt_cx, gt_cy, gt_bw, gt_bh = gt_boxes.unbind(-1)

        gt_x1 = (gt_cx - gt_bw / 2) * w
        gt_y1 = (gt_cy - gt_bh / 2) * h
        gt_x2 = (gt_cx + gt_bw / 2) * w
        gt_y2 = (gt_cy + gt_bh / 2) * h

        gt_boxes_xyxy = torch.stack(
            [gt_x1, gt_y1, gt_x2, gt_y2], dim=-1
        )

        ground_truth = {
            "boxes": gt_boxes_xyxy,
            "labels": target[i]["class_labels"]
        }

        metric.update([prediction], [ground_truth])

results = metric.compute()

print("mAP:", results["map"].item())
print("mAP@50:", results["map_50"].item())
print("mAP@75:", results["map_75"].item())

mAP: 0.04213859885931015
mAP@50: 0.08425817638635635
mAP@75: 0.0373440720140934


In [ ]:
import torch.nn as nn

fog_model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=8,
    ignore_mismatched_sizes=True
)

# Get the first convolution
conv = fog_model.model.backbone.model.conv1

# Create a new 4-channel convolution
new_conv = nn.Conv2d(
    in_channels=4,
    out_channels=conv.out_channels,
    kernel_size=conv.kernel_size,
    stride=conv.stride,
    padding=conv.padding,
    bias=False
)

# Copy pretrained RGB weights
with torch.no_grad():
    new_conv.weight[:, :3] = conv.weight

    # Start beta channel with zero weights
    new_conv.weight[:, 3:4] = 0

# Replace original convolution
fog_model.model.backbone.model.conv1 = new_conv

fog_model = fog_model.to(device)

print("Fog-aware DETR created!")
print("First conv:", fog_model.model.backbone.model.conv1)

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `91`.


Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |             

Fog-aware DETR created!
First conv: Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)


In [ ]:
from transformers import DetrForObjectDetection
import torch.nn as nn

fog_model = DetrForObjectDetection.from_pretrained(
    "/content/detr_baseline_best"
)

# Original trained first convolution
conv = fog_model.model.backbone.model.conv1

# Create 4-channel version
new_conv = nn.Conv2d(
    in_channels=4,
    out_channels=conv.out_channels,
    kernel_size=conv.kernel_size,
    stride=conv.stride,
    padding=conv.padding,
    bias=False
)

with torch.no_grad():
    # Keep all trained RGB weights
    new_conv.weight[:, :3] = conv.weight

    # New beta channel starts at zero
    new_conv.weight[:, 3:4] = 0

fog_model.model.backbone.model.conv1 = new_conv

fog_model = fog_model.to(device)

print("Fog-aware DETR initialized from trained baseline!")
print("First conv:", fog_model.model.backbone.model.conv1)

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

Fog-aware DETR initialized from trained baseline!
First conv: Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)


In [ ]:
def fog_collate_fn(batch):
    pixel_values = []

    for item in batch:
        rgb = item["pixel_values"]
        beta = item["beta"]

        # Create a beta channel matching the image dimensions
        beta_channel = torch.full(
            (1, rgb.shape[1], rgb.shape[2]),
            beta.item(),
            dtype=rgb.dtype
        )

        # RGB + beta
        pixel_values.append(
            torch.cat([rgb, beta_channel], dim=0)
        )

    pixel_values = torch.stack(pixel_values)

    pixel_mask = torch.stack([item["pixel_mask"] for item in batch])
    labels = [item["labels"][0] for item in batch]
    betas = torch.stack([item["beta"] for item in batch])

    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "labels": labels,
        "beta": betas
    }

fog_train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=fog_collate_fn
)

fog_val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=fog_collate_fn
)

print("Fog-aware DataLoaders ready!")

Fog-aware DataLoaders ready!


In [ ]:
batch = next(iter(fog_train_loader))

pixel_values = batch["pixel_values"].to(device)
pixel_mask = batch["pixel_mask"].to(device)

labels = [
    {k: v.to(device) for k, v in target.items()}
    for target in batch["labels"]
]

print("Input shape:", pixel_values.shape)
print("Beta values:", batch["beta"])

outputs = fog_model(
    pixel_values=pixel_values,
    pixel_mask=pixel_mask,
    labels=labels
)

print("Logits:", outputs.logits.shape)
print("Predicted boxes:", outputs.pred_boxes.shape)
print("Loss:", outputs.loss.item())

Input shape: torch.Size([4, 4, 666, 1333])
Beta values: tensor([0.0100, 0.0200, 0.0200, 0.0050])
Logits: torch.Size([4, 100, 9])
Predicted boxes: torch.Size([4, 100, 4])
Loss: 1.613250494003296


In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm

optimizer = AdamW(
    fog_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

num_epochs = 3
best_val_loss = float("inf")

for epoch in range(num_epochs):
    # ---- Training ----
    fog_model.train()
    train_loss = 0

    progress = tqdm(
        fog_train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for batch in progress:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)

        labels = [
            {k: v.to(device) for k, v in target.items()}
            for target in batch["labels"]
        ]

        optimizer.zero_grad()

        outputs = fog_model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(fog_train_loader)

    # ---- Validation ----
    fog_model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in fog_val_loader:
            pixel_values = batch["pixel_values"].to(device)
            pixel_mask = batch["pixel_mask"].to(device)

            labels = [
                {k: v.to(device) for k, v in target.items()}
                for target in batch["labels"]
            ]

            outputs = fog_model(
                pixel_values=pixel_values,
                pixel_mask=pixel_mask,
                labels=labels
            )

            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(fog_val_loader)

    print(
        f"Epoch {epoch+1}: "
        f"Train Loss = {avg_train_loss:.4f}, "
        f"Val Loss = {avg_val_loss:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        fog_model.save_pretrained("/content/detr_fog_best")
        processor.save_pretrained("/content/detr_fog_best")

        print("Saved best fog-aware model!")

Epoch 1/3:   0%|          | 0/374 [00:00<?, ?it/s]

Epoch 1: Train Loss = 1.9074, Val Loss = 2.2959


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best fog-aware model!


Epoch 2/3:   0%|          | 0/374 [00:00<?, ?it/s]

Epoch 2: Train Loss = 1.9363, Val Loss = 2.3234


Epoch 3/3:   0%|          | 0/374 [00:00<?, ?it/s]

Epoch 3: Train Loss = 1.8008, Val Loss = 2.1840


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best fog-aware model!


In [ ]:
fog_model.eval()

print("Using trained fog-aware model already in memory!")
print("First conv:", fog_model.model.backbone.model.conv1)

Using trained fog-aware model already in memory!
First conv: Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)


In [ ]:
fog_model = fog_model.to(device)
fog_model.eval()

print("Model device:", next(fog_model.parameters()).device)

Model device: cuda:0


In [ ]:
from tqdm.auto import tqdm

fog_predictions = []
fog_targets = []

with torch.no_grad():
    for batch in tqdm(fog_val_loader, desc="Running fog-aware validation"):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)

        outputs = fog_model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask
        )

        fog_predictions.append({
            "logits": outputs.logits.cpu(),
            "boxes": outputs.pred_boxes.cpu()
        })

        fog_targets.extend(batch["labels"])

print("Fog-aware validation inference complete!")
print("Batches:", len(fog_predictions))
print("Images:", len(fog_targets))

Running fog-aware validation:   0%|          | 0/39 [00:00<?, ?it/s]

Fog-aware validation inference complete!
Batches: 39
Images: 156


In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(
    box_format="xyxy",
    iou_type="bbox"
)

# Flatten targets
flat_targets = []
for batch in fog_targets:
    if isinstance(batch, list):
        flat_targets.extend(batch)
    else:
        flat_targets.append(batch)

for batch_pred, targets_batch in zip(fog_predictions, [
    flat_targets[i:i+4] for i in range(0, len(flat_targets), 4)
]):
    logits = batch_pred["logits"]
    boxes = batch_pred["boxes"]

    preds_list = []
    targets_list = []

    for i in range(len(logits)):
        scores = logits[i].softmax(-1)[:, :-1].max(-1).values
        labels = logits[i].softmax(-1)[:, :-1].argmax(-1)

        cx, cy, w, h = boxes[i].unbind(-1)

        pred_boxes = torch.stack([
            (cx - w / 2) * 2048,
            (cy - h / 2) * 1024,
            (cx + w / 2) * 2048,
            (cy + h / 2) * 1024
        ], dim=-1)

        preds_list.append({
            "boxes": pred_boxes,
            "scores": scores,
            "labels": labels
        })

        tcx, tcy, tw, th = targets_batch[i]["boxes"].unbind(-1)

        target_boxes = torch.stack([
            (tcx - tw / 2) * 2048,
            (tcy - th / 2) * 1024,
            (tcx + tw / 2) * 2048,
            (tcy + th / 2) * 1024
        ], dim=-1)

        targets_list.append({
            "boxes": target_boxes,
            "labels": targets_batch[i]["class_labels"]
        })

    metric.update(preds_list, targets_list)

results = metric.compute()

print("Fog-aware mAP:", results["map"].item())
print("Fog-aware mAP@50:", results["map_50"].item())
print("Fog-aware mAP@75:", results["map_75"].item())

Fog-aware mAP: 0.05597308278083801
Fog-aware mAP@50: 0.1167735755443573
Fog-aware mAP@75: 0.04879336431622505


In [ ]:
import pandas as pd

results_table = pd.DataFrame({
    "Model": ["Baseline DETR", "Fog-aware DETR", "Improvement"],
    "mAP": [0.0421, 0.0560, "+32.9%"],
    "mAP@50": [0.0843, 0.1168, "+38.8%"],
    "mAP@75": [0.0373, 0.0488, "+30.7%"]
})

results_table

,Model,mAP,mAP@50,mAP@75
0,Baseline DETR,0.0421,0.0843,0.0373
1,Fog-aware DETR,0.056,0.1168,0.0488
2,Improvement,+32.9%,+38.8%,+30.7%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/detr_baseline_best /content/drive/MyDrive/detr_baseline_best
!cp -r /content/detr_fog_best /content/drive/MyDrive/detr_fog_best

print("Model checkpoints backed up to Google Drive!")

Model checkpoints backed up to Google Drive!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

drive_path = "/content/drive/MyDrive"

print("Potential project files and folders:")
for item in os.listdir(drive_path):
    if any(word in item.lower() for word in [
        "fog", "cityscapes", "detr", "refined", "annotation"
    ]):
        print(item)

Mounted at /content/drive
Potential project files and folders:
detr_baseline_best
detr_fog_best


In [ ]:
drive_path = "/content/drive/MyDrive"

for folder in ["detr_baseline_best", "detr_fog_best"]:
    path = os.path.join(drive_path, folder)
    print(f"\n{folder}:")
    for item in os.listdir(path):
        print(" -", item)

print("\nSearching for dataset folders:")
for root, dirs, files in os.walk(drive_path):
    dirs[:] = [d for d in dirs if d not in [
        ".git", "node_modules", ".cache"
    ]]
    for folder in dirs:
        if any(word in folder.lower() for word in [
            "foggy_cityscapes", "cityscapes", "gtbboxvalidated"
        ]):
            print(os.path.join(root, folder))


detr_baseline_best:
 - config.json
 - model.safetensors
 - preprocessor_config.json

detr_fog_best:
 - config.json
 - preprocessor_config.json
 - model.safetensors

Searching for dataset folders:


In [ ]:
!pip install -q transformers timm pycocotools torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 26.3 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import DetrForObjectDetection, DetrImageProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

baseline_path = "/content/drive/MyDrive/detr_baseline_best"

processor = DetrImageProcessor.from_pretrained(baseline_path)

baseline_model = DetrForObjectDetection.from_pretrained(
    baseline_path
).to(device)

baseline_model.eval()

print("Device:", device)
print("Number of classes:", baseline_model.config.num_labels)
print("Backbone input channels:",
      baseline_model.model.backbone.model.conv1.in_channels)
print("Baseline checkpoint loaded successfully!")

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

Device: cuda
Number of classes: 8
Backbone input channels: 3
Baseline checkpoint loaded successfully!


In [ ]:
with torch.no_grad():
    backbone_output = baseline_model.model.backbone(
        pixel_values, pixel_mask
    )

print("Return type:", type(backbone_output))
print("Number of returned items:", len(backbone_output))

for i, item in enumerate(backbone_output):
    print(f"\nItem {i}:")
    print("Type:", type(item))

    if isinstance(item, (list, tuple)):
        print("Length:", len(item))
        for j, subitem in enumerate(item):
            if hasattr(subitem, "shape"):
                print(f"  [{j}] shape: {subitem.shape}")
            else:
                print(f"  [{j}] type: {type(subitem)}")
    elif hasattr(item, "shape"):
        print("Shape:", item.shape)
    else:
        print(item)

Return type: <class 'list'>
Number of returned items: 4

Item 0:
Type: <class 'tuple'>
Length: 2
  [0] shape: torch.Size([4, 256, 167, 334])
  [1] shape: torch.Size([4, 167, 334])

Item 1:
Type: <class 'tuple'>
Length: 2
  [0] shape: torch.Size([4, 512, 84, 167])
  [1] shape: torch.Size([4, 84, 167])

Item 2:
Type: <class 'tuple'>
Length: 2
  [0] shape: torch.Size([4, 1024, 42, 84])
  [1] shape: torch.Size([4, 42, 84])

Item 3:
Type: <class 'tuple'>
Length: 2
  [0] shape: torch.Size([4, 2048, 21, 42])
  [1] shape: torch.Size([4, 21, 42])


In [ ]:
import torch
import torch.nn as nn

class FiLMBackbone(nn.Module):
    def __init__(self, original_backbone, channels=2048):
        super().__init__()

        self.original_backbone = original_backbone

        # Convert the scalar fog density into feature-wise scale and shift.
        self.film_generator = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, channels * 2)
        )

        # Initially preserve the original features:
        # gamma = 1, delta = 0.
        final_layer = self.film_generator[-1]
        nn.init.zeros_(final_layer.weight)
        nn.init.zeros_(final_layer.bias)

        self.beta = None

    def forward(self, pixel_values, pixel_mask):
        features = self.original_backbone(pixel_values, pixel_mask)

        if self.beta is None:
            raise ValueError("Set the fog-density beta before forward pass.")

        beta = self.beta.to(
            device=pixel_values.device,
            dtype=pixel_values.dtype
        ).reshape(-1, 1)

        params = self.film_generator(beta)
        gamma_delta = params.chunk(2, dim=1)

        gamma = 1.0 + gamma_delta[0]
        delta = gamma_delta[1]

        # Preserve the earlier feature maps and modulate the final one.
        modulated = []
        for i, (feature_map, mask) in enumerate(features):
            if i == len(features) - 1:
                gamma = gamma.unsqueeze(-1).unsqueeze(-1)
                delta = delta.unsqueeze(-1).unsqueeze(-1)
                feature_map = gamma * feature_map + delta

            modulated.append((feature_map, mask))

        return modulated


# Keep a reference to the pretrained backbone and wrap it.
original_backbone = baseline_model.model.backbone

baseline_model.model.backbone = FiLMBackbone(
    original_backbone,
    channels=2048
).to(device)

print("FiLM backbone installed!")

FiLM backbone installed!


In [ ]:
# Get one validation batch
batch = next(iter(val_loader))

pixel_values = batch["pixel_values"].to(device)
pixel_mask = batch["pixel_mask"].to(device)
betas = batch["beta"].to(device)

# Pass fog density to the FiLM module
baseline_model.model.backbone.beta = betas

baseline_model.eval()

with torch.no_grad():
    outputs = baseline_model(
        pixel_values=pixel_values,
        pixel_mask=pixel_mask
    )

print("Input shape:", pixel_values.shape)
print("Fog densities:", betas.tolist())
print("Predicted logits:", outputs.logits.shape)
print("Predicted boxes:", outputs.pred_boxes.shape)
print("Forward pass successful!")

Input shape: torch.Size([4, 3, 666, 1333])
Fog densities: [0.004999999888241291, 0.004999999888241291, 0.009999999776482582, 0.019999999552965164]
Predicted logits: torch.Size([4, 100, 9])
Predicted boxes: torch.Size([4, 100, 4])
Forward pass successful!


In [ ]:
from transformers import DetrForObjectDetection
import torch

# Name the modified model clearly.
film_model = baseline_model
film_model.train()

# Fine-tune the pretrained detector and train the new FiLM module.
optimizer = torch.optim.AdamW(
    film_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

num_epochs = 3
best_val_loss = float("inf")

print("FiLM model ready for training")
print("Trainable parameters:",
      sum(p.numel() for p in film_model.parameters() if p.requires_grad))
print("Epochs:", num_epochs)
print("Learning rate:", 1e-5)

FiLM model ready for training
Trainable parameters: 41638669
Epochs: 3
Learning rate: 1e-05


In [ ]:
import os
import torch
from tqdm.auto import tqdm

film_model = film_model.to(device)

save_dir = "/content/drive/MyDrive/detr_film_best"
os.makedirs(save_dir, exist_ok=True)

best_val_loss = float("inf")
num_epochs = 3

for epoch in range(num_epochs):
    # Training
    film_model.train()
    train_loss_sum = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Train")

    for batch in train_bar:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        betas = batch["beta"].to(device)
        labels = [
            {k: v.to(device) for k, v in target.items()}
            for target in batch["labels"]
        ]

        film_model.model.backbone.beta = betas

        optimizer.zero_grad()

        outputs = film_model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss_sum / len(train_loader)

    # Validation
    film_model.eval()
    val_loss_sum = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            pixel_values = batch["pixel_values"].to(device)
            pixel_mask = batch["pixel_mask"].to(device)
            betas = batch["beta"].to(device)
            labels = [
                {k: v.to(device) for k, v in target.items()}
                for target in batch["labels"]
            ]

            film_model.model.backbone.beta = betas

            outputs = film_model(
                pixel_values=pixel_values,
                pixel_mask=pixel_mask,
                labels=labels
            )

            val_loss_sum += outputs.loss.item()

    avg_val_loss = val_loss_sum / len(val_loader)

    print(
        f"Epoch {epoch+1}: "
        f"train_loss={avg_train_loss:.4f}, "
        f"val_loss={avg_val_loss:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        # Save model weights and config.
        film_model.save_pretrained(save_dir)
        processor.save_pretrained(save_dir)

        # Save FiLM module weights separately because the wrapper
        # is custom and is not represented by the standard DETR config.
        torch.save(
            film_model.model.backbone.film_generator.state_dict(),
            os.path.join(save_dir, "film_generator.pt")
        )

        print(f"Saved best checkpoint to {save_dir}")

print("Training complete. Best validation loss:", best_val_loss)

Epoch 1/3 - Train:   0%|          | 0/374 [00:00<?, ?it/s]

Validation:   0%|          | 0/39 [00:00<?, ?it/s]

Epoch 1: train_loss=1.9126, val_loss=2.3796


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best checkpoint to /content/drive/MyDrive/detr_film_best


...

Epoch 3/3 - Train:   0%|          | 0/374 [00:00<?, ?it/s]

Validation:   0%|          | 0/39 [00:00<?, ?it/s]

Epoch 3: train_loss=1.7330, val_loss=2.0982


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best checkpoint to /content/drive/MyDrive/detr_film_best
Training complete. Best validation loss: 2.0981652430998974


In [ ]:
import torch
from tqdm.auto import tqdm

film_model.eval()
film_predictions = []
film_targets = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="FiLM validation inference"):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        betas = batch["beta"].to(device)

        film_model.model.backbone.beta = betas

        outputs = film_model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask
        )

        film_predictions.append({
            "logits": outputs.logits.cpu(),
            "boxes": outputs.pred_boxes.cpu()
        })

        film_targets.extend([
            {k: v.cpu() for k, v in target.items()}
            for target in batch["labels"]
        ])

print("Batches:", len(film_predictions))
print("Images:", len(film_targets))
print("Inference complete!")

FiLM validation inference:   0%|          | 0/39 [00:00<?, ?it/s]

Batches: 39
Images: 156
Inference complete!


In [ ]:

from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

for batch_idx, batch_pred in enumerate(film_predictions):
    logits = batch_pred["logits"]
    boxes = batch_pred["boxes"]

    # Matching ground-truth targets for this batch
    start = batch_idx * 4
    targets_batch = film_targets[start:start + len(logits)]

    preds_list = []
    targets_list = []

    for i in range(len(logits)):
        # Exclude DETR's final "no object" class
        probs = logits[i].softmax(-1)[:, :-1]
        scores = probs.max(-1).values
        labels = probs.argmax(-1)

        # Convert predicted boxes from normalized cxcywh to pixel xyxy
        cx, cy, w, h = boxes[i].unbind(-1)
        pred_boxes = torch.stack([
            (cx - w / 2) * 2048,
            (cy - h / 2) * 1024,
            (cx + w / 2) * 2048,
            (cy + h / 2) * 1024
        ], dim=-1)

        # Convert ground-truth boxes to pixel xyxy
        target = targets_batch[i]
        tcx, tcy, tw, th = target["boxes"].unbind(-1)
        target_boxes = torch.stack([
            (tcx - tw / 2) * 2048,
            (tcy - th / 2) * 1024,
            (tcx + tw / 2) * 2048,
            (tcy + th / 2) * 1024
        ], dim=-1)

        preds_list.append({
            "boxes": pred_boxes,
            "scores": scores,
            "labels": labels
        })

        targets_list.append({
            "boxes": target_boxes,
            "labels": target["class_labels"]
        })

    metric.update(preds_list, targets_list)

film_results = metric.compute()

print(f"FiLM mAP:    {film_results['map'].item():.4f}")
print(f"FiLM mAP@50: {film_results['map_50'].item():.4f}")
print(f"FiLM mAP@75: {film_results['map_75'].item():.4f}")


FiLM mAP:    0.0624
FiLM mAP@50: 0.1365
FiLM mAP@75: 0.0506
